<a href="https://colab.research.google.com/github/ZeinaDawod/NLP-PROJECT/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')
import os
folder_path = '/content/drive/MyDrive/training_data'
os.listdir(folder_path)

def load_data(base_folder):
    texts = []
    labels = []
    for label in ['pos', 'neg']:
        folder_path = os.path.join(base_folder, label)
        for filename in os.listdir(folder_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(folder_path, filename)
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()

                texts.append(text)
                labels.append(label)


    df = pd.DataFrame({'text': texts, 'label': labels})
    return df


df = load_data(folder_path)




print(df.head())

print(f"The Number Of Dec {len(df)}")
print(f"Positive: {len(df[df['label'] == 'pos'])}")
print(f"Negative: {len(df[df['label'] == 'neg'])}")



Mounted at /content/drive
                                                text label
0   " i know what you did last summer , " the fir...   pos
1  in _daylight_ , sylvester stallone breaks no n...   pos
2  there's a moment in schindler's list when a nu...   pos
3  a movie that's been as highly built up as the ...   pos
4  i actually am a fan of the original 1961 or so...   pos
The Number Of Dec 1654
Positive: 806
Negative: 848


# Data Preprocessing

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
df['text'] = df['text'].str.lower()

print("Lowercasing:")
print(df.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Lowercasing:
                                                text label
0   " i know what you did last summer , " the fir...   pos
1  in _daylight_ , sylvester stallone breaks no n...   pos
2  there's a moment in schindler's list when a nu...   pos
3  a movie that's been as highly built up as the ...   pos
4  i actually am a fan of the original 1961 or so...   pos


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df['text'] = df['text'].str.replace(r'<[^>]+>', '', regex=True)
df['text'] = df['text'].str.replace(r'http\S+|www\S+', '', regex=True)
df['text'] = df['text'].str.replace(r'[^a-z\s]', '', regex=True)
df['text'] = df['text'].str.strip()
print("Remove Noise:")
print(df.head())

Remove Noise:
                                                text label
0  i know what you did last summer   the first hi...   pos
1  in daylight  sylvester stallone breaks no new ...   pos
2  theres a moment in schindlers list when a numb...   pos
3  a movie thats been as highly built up as the t...   pos
4  i actually am a fan of the original  or so liv...   pos


In [ ]:
df['text'] = df['text'].apply(word_tokenize)
print("Tokenization:")
print(df.head())

Tokenization:
                                                text label
0  [i, know, what, you, did, last, summer, the, f...   pos
1  [in, daylight, sylvester, stallone, breaks, no...   pos
2  [theres, a, moment, in, schindlers, list, when...   pos
3  [a, movie, thats, been, as, highly, built, up,...   pos
4  [i, actually, am, a, fan, of, the, original, o...   pos


In [ ]:
stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda x: [word for word in x if word not in stop_words])
print("Stop Words Removal:")
print(df.head())

Stop Words Removal:
                                                text label
0  [know, last, summer, first, highprofile, slash...   pos
1  [daylight, sylvester, stallone, breaks, new, g...   pos
2  [theres, moment, schindlers, list, number, jew...   pos
3  [movie, thats, highly, built, truman, show, re...   pos
4  [actually, fan, original, liveactiondisney, fl...   pos


In [ ]:
stemmer = PorterStemmer()
df['text'] = df['text'].apply(lambda x: [stemmer.stem(word) for word in x])
print("Stemming:")
print(df.head())

Stemming:
                                                text label
0  [know, last, summer, first, highprofil, slashe...   pos
1  [daylight, sylvest, stallon, break, new, groun...   pos
2  [there, moment, schindler, list, number, jew, ...   pos
3  [movi, that, highli, built, truman, show, revi...   pos
4  [actual, fan, origin, liveactiondisney, flick,...   pos


In [ ]:
df['text'] = df['text'].apply(lambda x: ' '.join(x))
print(df.head())

                                                text label
0  know last summer first highprofil slasher thri...   pos
1  daylight sylvest stallon break new ground cine...   pos
2  there moment schindler list number jew trudg s...   pos
3  movi that highli built truman show review boas...   pos
4  actual fan origin liveactiondisney flick name ...   pos


In [ ]:
from collections import Counter


all_words = ' '.join(df['text']).split()
word_counts = Counter(all_words)
print(f"The Number Of Words Before {len(word_counts)}")


def remove_infrequent(text, min_count=2):
    words = text.split()
    filtered = [w for w in words if word_counts[w] >= min_count]
    return ' '.join(filtered)

df['text'] = df['text'].apply(remove_infrequent)


all_words_after = ' '.join(df['text']).split()
word_counts_after = Counter(all_words_after)
print(f"The Number Of Words After : {len(word_counts_after)}")
print(df.head())


The Number Of Words Before 28665
The Number Of Words After : 16908
                                                text label
0  know last summer first highprofil slasher thri...   pos
1  daylight sylvest stallon break new ground cine...   pos
2  there moment schindler list number jew trudg s...   pos
3  movi that highli built truman show review boas...   pos
4  actual fan origin liveactiondisney flick name ...   pos


# Feature Representation and Engineering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(df['text'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print("الـ TF-IDF Matrix:")
print(tfidf_df.round(3))
print()
print(f"Matrix: {tfidf_matrix.shape}")
print(f"The Number Of Dec {tfidf_matrix.shape[0]}")
print(f"The Num Of Words {tfidf_matrix.shape[1]}")

الـ TF-IDF Matrix:
      aaron  abandon  abbi  abduct  abil  abil make    abl  abl creat  \
0       0.0      0.0   0.0     0.0   0.0        0.0  0.026      0.054   
1       0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
2       0.0      0.0   0.0     0.0   0.0        0.0  0.037      0.000   
3       0.0      0.0   0.0     0.0   0.0        0.0  0.036      0.000   
4       0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
...     ...      ...   ...     ...   ...        ...    ...        ...   
1649    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1650    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1651    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1652    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1653    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   

      abl find  abl get  ...  zeta  zetajon  zinger  zip  zombi  zone  zoom  \
0          0.0      0.0  

### extra featuer

In [ ]:
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import MinMaxScaler
extra_features = []
for doc in df['text']:
    exclamation = doc.count('!')
    caps = sum(1 for w in doc.split() if w.isupper())
    length = len(doc.split())
    extra_features.append([exclamation, caps, length])

extra_matrix = np.array(extra_features)
scaler = MinMaxScaler()
extra_matrix_scaled = scaler.fit_transform(extra_matrix)
final_matrix = hstack([tfidf_matrix, extra_matrix_scaled])
print(f"Matrix: {final_matrix.shape}")

tfidf_columns = list(vectorizer.get_feature_names_out())
extra_columns  = ['exclamation', 'caps', 'length']
all_columns    = tfidf_columns + extra_columns

df_final = pd.DataFrame(
    final_matrix.toarray(),
    columns=all_columns
)
print("TF-IDF Matrix:")
print(df_final.round(3))


Matrix: (1654, 17529)
TF-IDF Matrix:
      aaron  abandon  abbi  abduct  abil  abil make    abl  abl creat  \
0       0.0      0.0   0.0     0.0   0.0        0.0  0.026      0.054   
1       0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
2       0.0      0.0   0.0     0.0   0.0        0.0  0.037      0.000   
3       0.0      0.0   0.0     0.0   0.0        0.0  0.036      0.000   
4       0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
...     ...      ...   ...     ...   ...        ...    ...        ...   
1649    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1650    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1651    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1652    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   
1653    0.0      0.0   0.0     0.0   0.0        0.0  0.000      0.000   

      abl find  abl get  ...  zip  zombi  zone  zoom  zorro  zucker  zwick  \
0       

# Model Training & Evaluation

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC(max_iter=5000),
    "Random Forest": RandomForestClassifier(),
    "Naive Bayes": ComplementNB(alpha=5.0)
}
params = {

    "Logistic Regression": {
        "C": [0.01,0.1,1,5]
    },

    "SVM": {
        "C": [0.01,0.05]


    },

    "Random Forest": {"n_estimators": [100],
    "max_depth": [3, 4],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [5, 10]
    },

    "Naive Bayes": {
       "alpha": [0.01,0.5,0.2, 1, 5]
    }
}







In [ ]:
print(f"\nTuning Logistic Regression...")

grid = GridSearchCV(
    models['Logistic Regression'],
    params['Logistic Regression'],
    cv=10,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1)

grid.fit(df_final,df['label'])
results = grid.cv_results_
for i in range(len(results["params"])):
    print("Params   :", results["params"][i])
    print("Train Score:", round(results["mean_train_score"][i], 3))
    print("CV Score :", round(results["mean_test_score"][i], 2))
    train_score = results["mean_train_score"][i]
    cv_score = results["mean_test_score"][i]
    if train_score - cv_score > 0.1:
        print("⚠ Possible Overfitting\n")
    print("-" * 30)


Tuning Logistic Regression...
Params   : {'C': 0.01}
Train Score: 0.535
CV Score : 0.53
------------------------------
Params   : {'C': 0.1}
Train Score: 0.828
CV Score : 0.74
------------------------------
Params   : {'C': 1}
Train Score: 0.973
CV Score : 0.83
⚠ Possible Overfitting

------------------------------
Params   : {'C': 5}
Train Score: 0.998
CV Score : 0.85
⚠ Possible Overfitting

------------------------------


In [ ]:
print(f"\nTuning Random Forest...")

grid = GridSearchCV(
    models['Random Forest'],
    params['Random Forest'],
    cv=10,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1)

grid.fit(df_final,df['label'])
results = grid.cv_results_
for i in range(len(results["params"])):
    print("Params   :", results["params"][i])
    print("Train Score:", round(results["mean_train_score"][i], 3))
    print("CV Score :", round(results["mean_test_score"][i], 2))
    train_score = results["mean_train_score"][i]
    cv_score = results["mean_test_score"][i]
    if train_score - cv_score > 0.1:
        print("⚠ Possible Overfitting\n")
    print("-" * 30)


Tuning Random Forest...
Params   : {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 10, 'n_estimators': 100}
Train Score: 0.862
CV Score : 0.77
------------------------------
Params   : {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 100}
Train Score: 0.86
CV Score : 0.77
------------------------------
Params   : {'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 100}
Train Score: 0.846
CV Score : 0.77
------------------------------
Params   : {'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 20, 'n_estimators': 100}
Train Score: 0.853
CV Score : 0.75
⚠ Possible Overfitting

------------------------------
Params   : {'max_depth': 4, 'min_samples_leaf': 5, 'min_samples_split': 10, 'n_estimators': 100}
Train Score: 0.887
CV Score : 0.78
⚠ Possible Overfitting

------------------------------
Params   : {'max_depth': 4, 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 100}
Train Score: 0

In [ ]:
print(f"\nTuning Naive Bayes...")


grid = GridSearchCV(
    models['Naive Bayes'],
    params['Naive Bayes'],
    cv=10,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1)

grid.fit(df_final,df['label'])
results = grid.cv_results_
for i in range(len(results["params"])):
    print("Params   :", results["params"][i])
    print("Train Score:", round(results["mean_train_score"][i], 3))
    print("CV Score :", round(results["mean_test_score"][i], 2))
    train_score = results["mean_train_score"][i]
    cv_score = results["mean_test_score"][i]
    if train_score - cv_score > 0.1:
        print("⚠ Possible Overfitting\n")
    print("-" * 30)


Tuning Naive Bayes...
Params   : {'alpha': 0.01}
Train Score: 0.987
CV Score : 0.81
⚠ Possible Overfitting

------------------------------
Params   : {'alpha': 0.5}
Train Score: 0.968
CV Score : 0.83
⚠ Possible Overfitting

------------------------------
Params   : {'alpha': 0.2}
Train Score: 0.975
CV Score : 0.83
⚠ Possible Overfitting

------------------------------
Params   : {'alpha': 1}
Train Score: 0.963
CV Score : 0.83
⚠ Possible Overfitting

------------------------------
Params   : {'alpha': 5}
Train Score: 0.942
CV Score : 0.81
⚠ Possible Overfitting

------------------------------


In [ ]:
print(f"\nTuning SVM...")

grid = GridSearchCV(
    models['SVM'],
    params['SVM'],
    cv=10,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1)

grid.fit(df_final,df['label'])
results = grid.cv_results_
for i in range(len(results["params"])):
    print("Params   :", results["params"][i])
    print("Train Score:", round(results["mean_train_score"][i], 3))
    print("CV Score :", round(results["mean_test_score"][i], 2))
    train_score = results["mean_train_score"][i]
    cv_score = results["mean_test_score"][i]
    if train_score - cv_score > 0.1:
        print("⚠️ Possible Overfitting\n")
    print("-" * 30)


Tuning SVM...
Params   : {'C': 0.01}
Train Score: 0.823
CV Score : 0.73
------------------------------
Params   : {'C': 0.05}
Train Score: 0.93
CV Score : 0.8
⚠️ Possible Overfitting

------------------------------


In [ ]:
print("\nModel Comparison (CV Scores):")

best_model_name = RandomForestClassifier(max_depth=3, min_samples_leaf=10, min_samples_split=20, n_estimators=100)
best_model_name.fit(df_final, df['label'])


Model Comparison (CV Scores):


RandomForestClassifier(max_depth=3, min_samples_leaf=10, min_samples_split=20)

In [ ]:
y_pred = best_model_name.predict(df_final)


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(df['label'], y_pred)
print(report)


              precision    recall  f1-score   support

         neg       0.80      0.96      0.87       848
         pos       0.94      0.75      0.83       806

    accuracy                           0.85      1654
   macro avg       0.87      0.85      0.85      1654
weighted avg       0.87      0.85      0.85      1654



In [ ]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(df['label'], y_pred, average="macro")
print("Macro F1-score:", macro_f1)


Macro F1-score: 0.851974122747243
